In [1]:
import yaml
import torch
import pytorch_lightning as pl
from tqdm.auto import tqdm
import torch.nn as nn
import numpy as np

In [2]:
import sys 
sys.path.append('../')
from src.spatial_attn_lightning import BinauralAttentionModule
from corpus.jsinV3AttnTrackingValidation import jsinV3_attn_tracking_validation
import src.audio_transforms as at

In [3]:
pl.seed_everything(1)

Global seed set to 1


1

In [6]:
path = 'config/binaural_attn/word_task_mixed_cue_large_architecture_v04.yml'
config = yaml.load(open(path, 'r'), Loader=yaml.FullLoader)

In [7]:
config['audio']['rep_kwargs']['rep_on_gpu'] = False

In [8]:
audio_config = config['audio']

audio_transforms = at.AudioCompose([
            at.AudioToTensor(),
            at.CombineWithRandomDBSNR(low_snr=0, high_snr=0), # set to 0 so foreground/background at same level 
            at.RMSNormalizeForegroundAndBackground(rms_level=0.1),
            at.UnsqueezeAudio(dim=0),
            at.AudioToAudioRepresentation(**audio_config),
])

center_crop=True
binaural=True
Binaural cochleagram
using IIR cochleagram


In [9]:
config['layernorm_first'] = True

model = BinauralAttentionModule(config)

dataset = jsinV3_attn_tracking_validation(**config['data']['corpus'],
                                          train=False,
                                          transform=audio_transforms,
                                          demo=False)

dataloader = torch.utils.data.DataLoader(
            dataset,
            batch_size=1,
            num_workers=10
        )

num_classes={'num_words': 800}
Model performing word task


AttributeError: module 'torch' has no attribute 'compile'